# Capítulo 9 — Subconsultas: Consultas dentro de Consultas

**Livro:** começa na página **343** do PDF.

**Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")
from banco import preparar
motor = preparar(9, "gabarito", recomecar=True)  # preparar(..., recomecar=True) volta o capítulo ao início
%reload_ext sql
%config SqlMagic.displaylimit = 60
%config SqlMagic.feedback = 0
%sql motor

## Greg utiliza uma conexão interior · PDF p. 346

#### Exercício: Primeiro, com junção

Mostre nome e cargo desejado de quem quer ser `Desenvolvedora Web` ou `Desenvolvedor Web` (junte `meus_contatos` e `emprego_desejado`).

In [ ]:
%%sql
SELECT mc.primeiro_nome, ed.cargo
FROM meus_contatos mc
JOIN emprego_desejado ed ON ed.contato_id = mc.contato_id
WHERE ed.cargo IN ('Desenvolvedora Web', 'Desenvolvedor Web');

## Subconsultas · PDF p. 348

#### Exercício: IN com subconsulta

Agora sem escrever os cargos à mão: quem deseja um cargo que aparece em `vagas` (`IN (SELECT cargo FROM vagas)`)?

In [ ]:
%%sql
SELECT mc.primeiro_nome, ed.cargo
FROM meus_contatos mc
JOIN emprego_desejado ed ON ed.contato_id = mc.contato_id
WHERE ed.cargo IN (SELECT cargo FROM vagas);

## Uma subconsulta em ação · PDF p. 351

#### Exercício: Subconsulta que devolve um valor

Mostre os contatos que moram em Natal usando `cep = (SELECT cep FROM codigos_cep WHERE cidade = 'Natal')`.

In [ ]:
%%sql
SELECT primeiro_nome, cep FROM meus_contatos
WHERE cep = (SELECT cep FROM codigos_cep WHERE cidade = 'Natal');

## Um acompanhamento na construção de uma subconsulta · PDF p. 355

#### Exercício: Construindo de dentro para fora

Quem ganha **acima da média** em `emprego_atual`? Rode antes só a subconsulta (`SELECT AVG(salario) ...`) numa célula nova para ver o número; depois use-a no `WHERE`.

In [ ]:
%%sql
SELECT mc.primeiro_nome, ea.cargo, ea.salario
FROM emprego_atual ea JOIN meus_contatos mc ON mc.contato_id = ea.contato_id
WHERE ea.salario > (SELECT AVG(salario) FROM emprego_atual)
ORDER BY ea.salario DESC;

## Uma subconsulta como uma coluna no comando SELECT · PDF p. 358

#### Exercício: Subconsulta no SELECT

Mostre o nome de cada contato e a cidade dele usando uma subconsulta **como coluna**: `(SELECT cidade FROM codigos_cep c WHERE c.cep = mc.cep) AS cidade`.

In [ ]:
%%sql
SELECT mc.primeiro_nome,
       (SELECT cidade FROM codigos_cep c WHERE c.cep = mc.cep) AS cidade
FROM meus_contatos mc;

## Outro exemplo: Subconsulta com uma conexão natural · PDF p. 359

#### Exercício: Quem ganha mais

Mostre nome e salário de quem tem o **maior** salário atual (`NATURAL JOIN` + `= (SELECT MAX(...))`).

In [ ]:
%%sql
SELECT primeiro_nome, salario
FROM meus_contatos NATURAL JOIN emprego_atual
WHERE salario = (SELECT MAX(salario) FROM emprego_atual);

## Uma subconsulta não-correlacionada com múltiplos valores: IN, NOT IN · PDF p. 364

#### Exercício: NOT IN

Quais profissões da tabela `profissao` não têm **nenhum** contato? Use `NOT IN (SELECT prof_id FROM meus_contatos)`.

In [ ]:
%%sql
SELECT profissao FROM profissao
WHERE prof_id NOT IN (SELECT prof_id FROM meus_contatos WHERE prof_id IS NOT NULL);

## Subconsultas correlacionadas · PDF p. 368

#### Exercício: Subconsulta correlacionada

Mostre os contatos com **exatamente 3** interesses. A subconsulta usa o `contato_id` da consulta de fora.

In [ ]:
%%sql
SELECT mc.primeiro_nome
FROM meus_contatos mc
WHERE 3 = (SELECT COUNT(*) FROM contato_interesse ci WHERE ci.contato_id = mc.contato_id);

## Uma subconsulta correlacionada (útil) com NOT EXISTS · PDF p. 368

#### Exercício: NOT EXISTS

Quem **não** tem emprego atual?

In [ ]:
%%sql
SELECT mc.primeiro_nome, mc.email
FROM meus_contatos mc
WHERE NOT EXISTS (SELECT 1 FROM emprego_atual ea WHERE ea.contato_id = mc.contato_id);

## EXISTS e NOT EXISTS · PDF p. 369

#### Exercício: EXISTS

Quem tem pelo menos um interesse entre `xadrez` e `games`?

In [ ]:
%%sql
SELECT mc.primeiro_nome
FROM meus_contatos mc
WHERE EXISTS (
    SELECT 1 FROM contato_interesse ci JOIN interesses i ON i.interesse_id = ci.interesse_id
    WHERE ci.contato_id = mc.contato_id AND i.interesse IN ('xadrez', 'games'));

## A empresa de Greg para serviços de recrutamento está aberta para negócios · PDF p. 370

#### Exercício: Candidatos para cada vaga

Para cada vaga, liste os contatos que desejam aquele cargo, estão disponíveis (`disponivel = 'S'`) e cujo `salario_min` cabe no salário da vaga.

In [ ]:
%%sql
SELECT v.cargo, v.salario, mc.primeiro_nome, ed.salario_min
FROM vagas v
JOIN emprego_desejado ed ON ed.cargo = v.cargo
JOIN meus_contatos mc ON mc.contato_id = ed.contato_id
WHERE ed.disponivel = 'S' AND ed.salario_min <= v.salario
ORDER BY v.cargo;